# SmartRetail AI — AI-Powered Supermarket Sales Analysis and Prediction

**Prepared by:** Vaishnavi Dipak Patil

This notebook explores supermarket sales, performs basic data-quality checks, creates summary tables and charts, and demonstrates a simple daily-sales forecasting baseline. Run from the project root folder. The dataset is included at `data/raw/supermarket_sales.csv`.

## 1. Import libraries

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

PROJECT_DIR = Path.cwd()
# If the notebook is launched from another folder, set PROJECT_DIR to the SmartRetail-AI folder.
DATA_PATH = PROJECT_DIR / 'data' / 'raw' / 'supermarket_sales.csv'
if not DATA_PATH.exists():
    alt = Path('supermarket_sales.csv')
    if alt.exists(): DATA_PATH = alt
print('Dataset path:', DATA_PATH.resolve())

## 2. Load and inspect the dataset

In [ ]:
df = pd.read_csv(DATA_PATH)
print('Shape:', df.shape)
display(df.head())
print('Columns:', df.columns.tolist())
display(df.dtypes.to_frame('data_type'))
display(df.isna().sum().to_frame('missing_values'))

## 3. Clean and prepare data

In [ ]:
df.columns = [c.strip() for c in df.columns]
if 'Date' in df.columns:
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
if 'Sales' in df.columns:
    df['Sales'] = pd.to_numeric(df['Sales'], errors='coerce')
if 'Quantity' in df.columns:
    df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')
if 'Unit Price' in df.columns:
    df['Unit Price'] = pd.to_numeric(df['Unit Price'], errors='coerce')
print('Duplicate rows:', int(df.duplicated().sum()))
print('Missing values after type conversion:')
display(df.isna().sum().to_frame('missing_values'))
clean_df = df.drop_duplicates().copy()
clean_df = clean_df.dropna(subset=['Date','Sales'])
print('Cleaned shape:', clean_df.shape)

## 4. Key business metrics

In [ ]:
total_sales = clean_df['Sales'].sum()
avg_transaction = clean_df['Sales'].mean()
total_units = clean_df['Quantity'].sum() if 'Quantity' in clean_df.columns else np.nan
avg_rating = clean_df['Rating'].mean() if 'Rating' in clean_df.columns else np.nan
metrics = pd.DataFrame({'Metric':['Rows after cleaning','Total sales','Average transaction sales','Total units sold','Average rating','Start date','End date'], 'Value':[len(clean_df), round(total_sales,2), round(avg_transaction,2), total_units, round(avg_rating,2) if pd.notna(avg_rating) else None, clean_df['Date'].min().date(), clean_df['Date'].max().date()]})
display(metrics)

## 5. Sales by category, city and product

In [ ]:
if 'Category' in clean_df.columns:
    category_sales = clean_df.groupby('Category', dropna=False)['Sales'].sum().sort_values(ascending=False)
    display(category_sales.rename('Total Sales').to_frame())
    category_sales.plot(kind='bar', title='Total Sales by Category', ylabel='Sales', xlabel='Category', figsize=(9,4), rot=35)
    plt.tight_layout(); plt.show()
if 'City' in clean_df.columns:
    city_sales = clean_df.groupby('City', dropna=False)['Sales'].sum().sort_values(ascending=False)
    display(city_sales.rename('Total Sales').to_frame())
    city_sales.plot(kind='bar', title='Total Sales by City', ylabel='Sales', xlabel='City', figsize=(8,4), rot=0)
    plt.tight_layout(); plt.show()
if 'Product' in clean_df.columns:
    product_sales = clean_df.groupby('Product', dropna=False)['Sales'].sum().sort_values(ascending=False)
    display(product_sales.head(10).rename('Total Sales').to_frame())

## 6. Daily sales trend

In [ ]:
daily_sales = clean_df.groupby('Date')['Sales'].sum().sort_index().asfreq('D', fill_value=0)
ax = daily_sales.plot(figsize=(11,4), title='Daily Sales Trend', ylabel='Sales', xlabel='Date')
plt.tight_layout(); plt.show()
print('Number of calendar days:', len(daily_sales))

## 7. Baseline 14-day sales forecast

In [ ]:
# A small dataset with intermittent transaction dates makes forecasting uncertain.
# This is an educational baseline, not a guarantee of future sales.
series = daily_sales.astype(float)
if len(series) >= 35:
    supervised = pd.DataFrame({'sales': series})
    for lag in [1,2,3,7,14]: supervised[f'lag_{lag}'] = supervised['sales'].shift(lag)
    supervised = supervised.dropna()
    features = [c for c in supervised.columns if c.startswith('lag_')]
    split = max(1, int(len(supervised)*0.8))
    train, test = supervised.iloc[:split], supervised.iloc[split:]
    model = RandomForestRegressor(n_estimators=200, random_state=42, min_samples_leaf=2)
    model.fit(train[features], train['sales'])
    if len(test):
        pred = model.predict(test[features])
        print('Chronological holdout MAE:', round(mean_absolute_error(test['sales'], pred),2))
        print('Chronological holdout RMSE:', round(mean_squared_error(test['sales'], pred)**0.5,2))
    history = list(series.values)
    future_dates = pd.date_range(series.index.max()+pd.Timedelta(days=1), periods=14, freq='D')
    future_values=[]
    for dt in future_dates:
        row = pd.DataFrame([[history[-1], history[-2], history[-3], history[-7], history[-14]]], columns=features)
        value = max(0.0, float(model.predict(row)[0]))
        future_values.append(value); history.append(value)
    forecast = pd.DataFrame({'Date':future_dates, 'Forecast_Sales':future_values})
    display(forecast.round(2))
    forecast.set_index('Date')['Forecast_Sales'].plot(figsize=(10,4), marker='o', title='Baseline 14-Day Sales Forecast', ylabel='Forecast sales')
    plt.tight_layout(); plt.show()
else:
    print('Not enough daily observations for the configured baseline model.')

## 8. Save cleaned data and summary outputs

In [ ]:
output_dir = PROJECT_DIR / 'outputs'
output_dir.mkdir(exist_ok=True)
clean_dir = PROJECT_DIR / 'data' / 'processed'
clean_dir.mkdir(parents=True, exist_ok=True)
clean_df.to_csv(clean_dir / 'supermarket_sales_clean.csv', index=False)
if 'Category' in clean_df.columns: clean_df.groupby('Category')['Sales'].sum().sort_values(ascending=False).to_csv(output_dir / 'sales_by_category.csv')
if 'City' in clean_df.columns: clean_df.groupby('City')['Sales'].sum().sort_values(ascending=False).to_csv(output_dir / 'sales_by_city.csv')
print('Saved cleaned data and available summary tables.')

## 9. Findings and limitations

- Use category, city, and product summaries to identify strong sales areas.
- Use the daily trend to see changes over time.
- Forecast values are estimates and should be interpreted cautiously because the dataset is limited and sales may be sparse across dates.
- Validate insights with more historical data before making operational or financial decisions.